In [ ]:
import spacy
from spacy.tokens import DocBin
from spacy.training.example import Example
from spacy.scorer import Scorer
import random
from IPython.display import HTML, display
from spacy import displacy

In [13]:
spacy.prefer_gpu()

False

In [ ]:
# todo
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 50.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 735.6/735.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [en-core-web-trf] [en-core-web-trf]
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [7]:
# defaults
! python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency  --force

ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [ ]:
!python -m spacy debug data config.cfg --paths.train ./training_1.spacy --paths.dev ./test.spacy

In [ ]:
nlp = spacy.blank('en')

! python -m spacy train config.cfg --output ./ --paths.train ./training_1.spacy --paths.dev ./test.spacy

ℹ Saving to output directory: .
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00    243.77    0.28    0.21    0.42    0.00
  0     200       2260.13   8361.60    0.00    0.00    0.00    0.00
  0     400       2000.92   4711.97    0.19    0.52    0.12    0.00
  0     600       3609.30   5354.27    0.15    0.87    0.08    0.00
  0     800        926.62   4181.62    0.05    2.37    0.02    0.00
  0    1000        808.62   4134.44    0.00    0.00    0.00    0.00
  0    1200       2128.43   4500.84    0.09    0.67    0.05    0.00
  0    1400       3053.89   3585.74    0.07    1.51    0.04    0.00
  0    1600        417.36   3811.18 

In [3]:
nlp_trained = spacy.load("model-best")

In [ ]:
def evaluate_spacy(nlp):
    doc_bin = DocBin().from_disk('test.spacy')
    docs = list(doc_bin.get_docs(nlp.vocab))

    examples = [Example.from_dict(doc, {
        "entities":[(ent.start_char,ent.end_char,ent.label_) for ent in doc.ents]
    })
    for doc in docs]

    scorer = Scorer()
    for e in examples:
        prediction = nlp(e.reference.text)
        ex = Example(prediction, e.reference)

    scores = scorer.score(examples)
    return docs, scores

In [ ]:
docs, scores = evaluate_spacy(nlp_trained)


In [24]:
print(
    f'precision: {scores["ents_p"]}\n'+
    f'recall: {scores["ents_r"]}\n'+
    f'f-score: {scores["ents_f"]}'
)

precision: 1.0
recall: 1.0
f-score: 1.0


In [31]:
# display example data
seed = 4654123
random.seed(seed)
sample = random.choice(docs)
sample_pred = nlp_trained(sample.text)



In [32]:
html = displacy.render(sample, style='ent', jupyter=False)
display(HTML(html))